# Module 2 — Heatwave Risk Model for Indian Real Estate

This notebook builds a heatwave-risk classifier for 15 Indian cities using
daily maximum temperature data from the **Open-Meteo Archive API** (2015-2023).

**Pipeline:**
1. Fetch daily `temperature_2m_max` and `apparent_temperature_max` for each city
2. Engineer yearly heatwave features (days > 40 °C, > 45 °C, max temp, heat-index days, consecutive hot days)
3. Label years as heatwave-affected (`days_above_40c > 10`)
4. Train an XGBoost classifier (avoiding leakage by excluding `days_above_40c`)
5. Produce city-level heatwave risk scores (0-100)


In [ ]:
# ── Cell 1: Fetch temperature data from Open-Meteo ────────────────────────
import requests, time, pandas as pd, numpy as np, os, warnings
warnings.filterwarnings('ignore')

CITIES = [
    {"name": "Jodhpur",    "lat": 26.24, "lon": 73.02},
    {"name": "Jaisalmer",  "lat": 26.92, "lon": 70.91},
    {"name": "Bikaner",    "lat": 28.02, "lon": 73.31},
    {"name": "Nagpur",     "lat": 21.15, "lon": 79.09},
    {"name": "Pune",       "lat": 18.52, "lon": 73.86},
    {"name": "Chennai",    "lat": 13.08, "lon": 80.27},
    {"name": "Bengaluru",  "lat": 12.97, "lon": 77.59},
    {"name": "Hyderabad",  "lat": 17.39, "lon": 78.49},
    {"name": "Bhopal",     "lat": 23.26, "lon": 77.41},
    {"name": "Patna",      "lat": 25.59, "lon": 85.14},
    {"name": "Mumbai",     "lat": 19.08, "lon": 72.88},
    {"name": "Kolkata",    "lat": 22.57, "lon": 88.36},
    {"name": "Guwahati",   "lat": 26.14, "lon": 91.74},
    {"name": "Varanasi",   "lat": 25.32, "lon": 82.97},
    {"name": "Srinagar",   "lat": 34.08, "lon": 74.80},
]

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

frames = []
for city in CITIES:
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "start_date": "2015-01-01",
        "end_date": "2023-12-31",
        "daily": "temperature_2m_max,apparent_temperature_max",
        "timezone": "Asia/Kolkata",
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    data = resp.json()
    df = pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "temp_max_c": data["daily"]["temperature_2m_max"],
        "apparent_temp_max_c": data["daily"]["apparent_temperature_max"],
    })
    df["city"] = city["name"]
    df["lat"]  = city["lat"]
    df["lon"]  = city["lon"]
    frames.append(df)
    print(f"  ✓ {city['name']} — {len(df)} rows")
    time.sleep(0.3)

df_all = pd.concat(frames, ignore_index=True)
df_all.fillna(0, inplace=True)
print(f"\nTotal rows: {len(df_all)}")
df_all.head()


In [ ]:
# ── Cell 2: Feature engineering (yearly aggregation) ──────────────────────
df_all["year"] = df_all["date"].dt.year

def compute_consecutive_hot_days(series, threshold=40):
    """Longest streak of consecutive days where temp > threshold."""
    is_hot = series > threshold
    if not is_hot.any():
        return 0
    return int(is_hot.groupby((~is_hot).cumsum()).sum().max())

yearly = df_all.groupby(["city", "year"]).agg(
    days_above_40c        = ("temp_max_c",          lambda x: (x > 40).sum()),
    days_above_45c        = ("temp_max_c",          lambda x: (x > 45).sum()),
    max_temp_year         = ("temp_max_c",          "max"),
    heat_index_days       = ("apparent_temp_max_c", lambda x: (x > 45).sum()),
    consecutive_hot_days  = ("temp_max_c",          lambda s: compute_consecutive_hot_days(s)),
    lat                   = ("lat",                 "first"),
    lon                   = ("lon",                 "first"),
).reset_index()

print(f"Yearly feature rows: {len(yearly)}")
yearly.head(10)


In [ ]:
# ── Cell 3: Heatwave label ────────────────────────────────────────────────
yearly["heatwave_label"] = (yearly["days_above_40c"] > 10).astype(int)

print("Label distribution:")
print(yearly["heatwave_label"].value_counts())
print(f"\nPositive rate: {yearly['heatwave_label'].mean():.2%}")


In [ ]:
# ── Cell 4: Train XGBoost model ──────────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# Exclude days_above_40c to avoid label leakage
FEATURES = ["days_above_45c", "max_temp_year", "heat_index_days", "consecutive_hot_days"]
TARGET   = "heatwave_label"

train = yearly[yearly["year"] <= 2021]
test  = yearly[yearly["year"] >= 2022]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print("── Test-Set Metrics ──")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
try:
    auc = roc_auc_score(y_test, y_proba[:, 1])
    print(f"AUC-ROC   : {auc:.4f}")
except ValueError:
    print("AUC-ROC   : N/A (single class in test set)")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1        : {f1_score(y_test, y_pred, zero_division=0):.4f}")

# Feature importance plot
os.makedirs("../data/outputs", exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4))
importances = model.feature_importances_
idx = np.argsort(importances)
ax.barh([FEATURES[i] for i in idx], importances[idx], color="#e74c3c")
ax.set_xlabel("Importance")
ax.set_title("Heatwave Model — Feature Importance")
plt.tight_layout()
fig.savefig("../data/outputs/heatwave_feature_importance.png", dpi=150)
plt.show()
print("Saved: ../data/outputs/heatwave_feature_importance.png")


In [ ]:
# ── Cell 5: Risk scores ──────────────────────────────────────────────────
all_proba = model.predict_proba(yearly[FEATURES])[:, 1]
yearly["heatwave_prob"] = all_proba
yearly["heatwave_risk_score"] = np.round(all_proba * 100, 2)

def categorize_risk(score):
    if score <= 25:
        return "LOW"
    elif score <= 50:
        return "MEDIUM"
    elif score <= 75:
        return "HIGH"
    else:
        return "VERY HIGH"

yearly["risk_category"] = yearly["heatwave_risk_score"].apply(categorize_risk)

risk_scores = yearly[["city", "year", "lat", "lon", "heatwave_label",
                       "heatwave_prob", "heatwave_risk_score", "risk_category"]].copy()

print(risk_scores.groupby("risk_category")["city"].count())
risk_scores.head(15)


In [ ]:
# ── Cell 6: Save outputs ─────────────────────────────────────────────────
import joblib

os.makedirs("../data/outputs", exist_ok=True)

risk_scores.to_csv("../data/outputs/heatwave_risk_scores.csv", index=False)
joblib.dump(model, "../data/outputs/heatwave_model.pkl")

print("Saved:")
print("  • ../data/outputs/heatwave_risk_scores.csv")
print("  • ../data/outputs/heatwave_model.pkl")
print(f"\nRisk-score table shape: {risk_scores.shape}")
